<a href="https://colab.research.google.com/github/TianmingZhou1963/My-web/blob/master/975_WEEK_2_lab_cw_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# this mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# enter the foldername in your Drive where you have saved the unzipped
# lab folder, e.g. 'UOW/AISecurity/lab_cw_attack/'
FOLDERNAME = 'UOW/AISecurity/lab_cw_attack/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))
sys.path.append('/content/drive/My Drive/{}/codebase'.format(FOLDERNAME))

%cd /content

In [ ]:
import torch
import sys

device = torch.device('cuda' if torch.cuda.is_available else 'cpu')

print('PyTorch Version:', torch.__version__)
print('-' * 60)
if torch.cuda.is_available():
    print('CUDA Device Count:', torch.cuda.device_count())
    print('CUDA Device Name:')
    for i in range(torch.cuda.device_count()):
        print('\t', torch.cuda.get_device_name(i))
    print('CUDA Current Device Index:', torch.cuda.current_device())
    print('-' * 60)

print(f"Python version = {sys.version}")


In [ ]:
# As usual, a bit of setup
import matplotlib.pyplot as plt
import types
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

exp_cfg = types.SimpleNamespace()
exp_cfg.data_dir = Path(f"/content/drive/My Drive/{FOLDERNAME}/data")
exp_cfg.out_dir = Path(f"/content/drive/My Drive/{FOLDERNAME}/out")

exp_cfg.data_dir.mkdir(parents=True, exist_ok=True)
exp_cfg.out_dir.mkdir(parents=True, exist_ok=True)

exp_cfg.device = torch.device('cuda:0')  # use the first GPU


# Preparing Models

Let's train 2 VGG models on CIFAR10.
This may take around 40 minutes if a Tesla T4 GPU is assigned.

Please read codebase/attacks/cw_l2.py during this time to understand how CW l2 attack is implemented.

If you don't want to wait for the training, you can download pre-trained models from https://uowmailedu-my.sharepoint.com/:u:/g/personal/wzong_uow_edu_au/EWvH2WUYvptJreIN50IH4joBnE37JF_pwZrvfulj3j-PTw?e=KR8Xo2

Unzip and upload the "out" folder to your lab folder.

CW attack paper link: https://arxiv.org/abs/1608.04644

Formula for CW $\ell_2$ attack can be found in Section IV.


In [ ]:
from codebase import model_trainer, utils, setup
import torchvision
import torchvision.transforms as transforms
from codebase.classifiers import vgg

# Download CIFAR10 dataset from pytorch.
exp_cfg.out_dir.mkdir(parents=True, exist_ok=True)

cifar10_mean_tensor = torch.Tensor(setup.CIFAR10_MEAN).reshape([1, 3, 1, 1]).to(exp_cfg.device)
cifar10_std_tensor = torch.Tensor(setup.CIFAR10_STD).reshape([1, 3, 1, 1]).to(exp_cfg.device)

# feel free to add more augmentation to boost the performance.
train_set = torchvision.datasets.CIFAR10(root=str(exp_cfg.data_dir), train=True, download=True,
                                            transform=transforms.Compose([
                                                    transforms.ToTensor(),
                                                    transforms.RandomHorizontalFlip(0.5),
                                                    transforms.Normalize(setup.CIFAR10_MEAN, setup.CIFAR10_STD)
                                                ])
                                            )

test_set = torchvision.datasets.CIFAR10(root=str(exp_cfg.data_dir), train=False,
                                        download=True, transform=transforms.Compose([
                                            transforms.ToTensor(),
                                            transforms.Normalize(setup.CIFAR10_MEAN, setup.CIFAR10_STD)
                                            ])
                                        )

# there are 10 classes.
num_classes = len(setup.CIFAR10_CLASSES)

In [ ]:
# Train 2 classifiers from scratch.

model_num = 2
model_lst = []
for i in range(model_num):
    trainer_cfg = model_trainer.ModelTrainerConfig(
        batch_size=128, test_batch_size=128,
        num_workers=8,
        test_every=None,
        train_drop_last=True,
        loss_func=model_trainer.cross_entropy_loss,
        is_classifier=True,
        max_epochs=50,
        lr=1e-3,
        lr_gamma=0.1,
        lr_step_size=[40],  # the learning rate goes 1e-4 after 40 epochs
        ckpt_dir=exp_cfg.out_dir.joinpath(f"cpkt_model_{i}"),
    )

    # use vgg11
    model = vgg.vgg11_bn(num_classes=10).to(exp_cfg.device)
    trainer = model_trainer.ModelTrainer(model=model, train_set=train_set,
                                            test_set=test_set, config=trainer_cfg)

    # Train the model if saved weights cannot be found.
    trainer.run()
    model_lst.append(model)

    print(f"Evaluating model {i}...")
    trainer.eval(trainer.test_loader, "testing set")



# CW $\ell_2$ Attack

This attack was originally proposed by Carlini and Wagner.
It is a strong iterative attack that finds adversarial examples on many defenses that are robust to other attacks.
CW $\ell_2$ attack is an iterative attack.
It is significantly slower than FGSM.


In this lab task, you will experiment with this attack.
You will also see whether this attack can transfer to other models that are trained in the same way but have different weights.

Now we have trained 3 models.
We use the first model to generate attacks.
The other models are kept for evaluating transferability.

We consider 3 different confidence values for CW $\ell_2$ attack:
- confidence = 1
- confidence = 15
- confidence = 50

We select 32 clean images that are correctly classified by the first model.
For each image, we then generate 3 adversarial examples corresponding to 3 confidence values.

In [ ]:
import numpy as np

first_model = model_lst[0]
first_model.eval()

num_adv = 32    # number of adversarial examples to generate
x_arr = []      # store clean data that can be correctly recognized
y_arr = []      # store the ground truth labels for these clean data

# fix the random state to make results repeatable
rand_state = np.random.RandomState(42)
rand_idxes = rand_state.permutation(len(test_set))

# randomly get images that are correctly recognized by the model
for idx in rand_idxes:
    # randomly get one
    test_x, test_y = test_set[idx]
    test_x = test_x.to(exp_cfg.device)
    assert isinstance(test_y, int)  # test_x is Tensor while test_y is simply an integer

    pred = first_model(test_x.unsqueeze(0)).argmax(dim=1)
    if pred.item() == test_y:
        x_arr.append(test_x)
        y_arr.append(test_y)

    if len(x_arr) >= num_adv:
        break
assert len(x_arr) == num_adv, "Cannot find enough correctly predicted clean data."

x_arr = torch.stack(x_arr)
y_arr = torch.LongTensor(y_arr).to(exp_cfg.device)
target_arr = (y_arr + 1) % num_classes       # choose target label as the next label.

org_logits = first_model(x_arr)

# Let's see the difference between the largest and the second largest logit values
y_onehot = torch.nn.functional.one_hot(y_arr, num_classes).float()
largest_logit = torch.sum(y_onehot * org_logits, 1)
sec_logit, _ = torch.max((1 - y_onehot) * org_logits - y_onehot * 1e4, 1)

log_diff = largest_logit - sec_logit
print(f"Difference between the largest and the second largest logit values (mean = {log_diff.mean()}):")
print(f"{log_diff}")

org_preds = org_logits.argmax(dim=1)
assert (org_preds == y_arr).sum().item() == num_adv, "Some images are incorrectly classified!"

In [ ]:
from codebase.attacks.cw_l2 import carlini_wagner_l2

# 32 clean images are saved in x_arr.
# Generate targeted adversarial examples using CW l2 attack.

# This dictionary saves adversarial examples corresponding to different confidence.
adv_dict = {1: None, 40: None, 80: None}

# save which attacks are successful for all the confidence values
succ_mask = torch.ones(len(x_arr), dtype=torch.bool).to(exp_cfg.device)

for conf_val in adv_dict.keys():

    print(f"Generate CW attacks for confidence = {conf_val}...")

    # x_arr is already normalized. However, input to carlini_wagner_l2 should not be normalized.
    org_x_arr = utils.unnormalize(x_arr, cifar10_mean_tensor, cifar10_std_tensor)

    # Our model accepts normalized input.
    model_fn = lambda _x: first_model(utils.normalize(_x, cifar10_mean_tensor, cifar10_std_tensor))

    adv_examples = carlini_wagner_l2(
        model_fn=model_fn,
        x=org_x_arr,
        n_classes=num_classes,
        y=target_arr,
        targeted=True,
        lr=1e-2,
        confidence=conf_val,
        clip_min=0,
        clip_max=1,
        initial_const=1.0,
        max_iterations=1000,
    )

    # Our model accepts normalized input.
    adv_examples = utils.normalize(adv_examples, cifar10_mean_tensor, cifar10_std_tensor)

    # Calculate the attack success rates and average confidence probability.
    adv_probs = torch.softmax(first_model(adv_examples), dim=1)

    target_onehot = torch.nn.functional.one_hot(target_arr, num_classes).float()
    attack_probs = torch.sum(target_onehot * adv_probs, 1)

    adv_preds = adv_probs.argmax(dim=1)
    succ = (adv_preds == target_arr)
    succ_num = succ.sum().item()
    succ_probs = attack_probs[succ]

    succ_mask = torch.logical_and(succ_mask, succ)

    print(f"CW l2 attacks (confidence {conf_val}) success = {succ_num} / {num_adv}; "
            f"prediction probability = {torch.mean(succ_probs).item()*100:.2f}%.")

    # save normalized adversarial examples for later use
    adv_dict[conf_val] = adv_examples

# Visualization

Visualize adversarial examples for each confidence value.

In [ ]:
succ_idx = []
for idx, mask in enumerate(succ_mask):
    if mask.item() is True:
        succ_idx.append(idx)
print(f"Managed to use {len(succ_idx)} clean images to generate adversarial examples for all confidence values.")
assert len(succ_idx) >= num_adv * 0.9, "CW l2 attack is very strong. Try larger values for 'initial_const' or 'lr' if success rates are below 90%."

for conf_val, adv_examples in adv_dict.items():

    print(f"Visualize CW attacks for confidence = {conf_val}.")

    for i in succ_idx[:1]:
        org_img = utils.unnormalize(x_arr[i], mean_vals=cifar10_mean_tensor.squeeze(0), std_vals=cifar10_std_tensor.squeeze(0))
        # To plot images, we change pixel values from [0, 1] to [0, 255]
        # We also change the dimensions from [channel, height, width] to [height, width, channel]
        org_img = (org_img*255).detach().cpu().numpy().astype(np.uint8).transpose([1, 2, 0])
        org_label = y_arr[i].item()

        adv_img = utils.unnormalize(adv_examples[i], mean_vals=cifar10_mean_tensor.squeeze(0), std_vals=cifar10_std_tensor.squeeze(0))
        adv_img = (adv_img*255).detach().cpu().numpy().astype(np.uint8).transpose([1, 2, 0])
        adv_target = target_arr[i].item()

        plt.subplot(1, 4, 1)
        plt.imshow(org_img)
        plt.title(setup.CIFAR10_CLASSES[org_label])
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.imshow(adv_img)
        plt.title(setup.CIFAR10_CLASSES[adv_target])
        plt.axis('off')

        plt.subplot(1, 4, 3)
        abs_diff = np.abs(adv_img.astype(np.int32) - org_img.astype(np.int32))
        l2_norm = (abs_diff**2).sum()**0.5 / 255
        l_inf_norm = abs_diff.max() / 255
        plt.imshow(abs_diff.astype(np.uint8))
        plt.title(f'Diff (l_2 = {l2_norm:.2f}; l_inf = {l_inf_norm:.2f})')
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.imshow((abs_diff*10).astype(np.uint8))
        plt.title('Magnified difference (10x)')
        plt.axis('off')

        plt.gcf().set_size_inches(12, 5)
        plt.show()
        plt.close()

# Transferability

Can CW attacks transfer to other models?

In [ ]:
other_models_lst = model_lst[1:]    # the first model was used for generating attacks

for idx, other_model in enumerate(other_models_lst):
    other_model.eval()

    print(f"Evaluating other_model {idx}:")

    # make predictions on the clean data
    org_preds = other_model(x_arr).argmax(dim=1)
    correct = (org_preds == y_arr).sum()
    print(f"Correct predictions on clean data = {correct} / {len(y_arr)}")

    # make predictions on the adversarial examples
    for conf_val, adv_examples in adv_dict.items():

        print(f"\n***** Evaluating robustness against adversarial examples with confidence = {conf_val}) *****\n")

        ae_preds = other_model(adv_examples).argmax(dim=1)
        ae_correct = (ae_preds == y_arr).sum()
        fooling = (ae_preds == target_arr).sum()

        print(f"Correct predictions on adversarial examples = {ae_correct} / {len(y_arr)}")
        print(f"Targeted fooling rate = {fooling} / {len(y_arr)}")